# Address Cleaning

In [ ]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder.master("local").appName("test").getOrCreate()

# Read CSV file
df = spark.read.csv(r"test_files/test_1.csv")

## Pre-Processing Examples

### Clean Punctuation

Cleans up punctuation from address strings by removing or fixing unwanted characters, while preserving hyphens and periods where necessary (e.g., between numbers or block names).

In [ ]:
from address_cleaning.functions.pre_processing import clean_punctuation

cleaned_punctation_df = clean_punctuation(df, '_c1', create_flag = True, overwrite = False)
cleaned_punctation_df.filter(cleaned_punctation_df.punctuation_cleaned_flag == 1).show(truncate = False)

### Remove Noise Words

Removes noise words from the input address column. Noise words are defined as sequences of the same uppercase letter repeated three or more times (e.g., "AAAA")

In [ ]:
from address_cleaning.functions.pre_processing import remove_noise_words_with_flag

noise_removed_df = remove_noise_words_with_flag(df, '_c1', create_flag=True, overwrite=True)
noise_removed_df.filter(noise_removed_df.noise_removed_flag == 1).show(truncate = False)

### Deduplicate Addresses (REVIEW CHANGES)

Processes and deduplicates parts of an address based on similarity of consecutive parts.

Before:

In [ ]:
from address_cleaning.functions.pre_processing import get_process_and_deduplicate_address_udf

deduplicate_addresses_udf = get_process_and_deduplicate_address_udf()
deduplicated_addresses = deduplicate_addresses_udf(df["_c1"])["cleaned_address"]
deduplicated_addresses_flag = deduplicate_addresses_udf(df["_c1"])["words_deduplicated_flag"]


deduplicate_addresses_df = df.withColumn("cleaned", deduplicated_addresses)
deduplicate_addresses_df = deduplicate_addresses_df.withColumn("words_deduplicated_flag", deduplicated_addresses_flag)

deduplicate_addresses_df.filter(deduplicate_addresses_df.words_deduplicated_flag == 1).show(truncate = False)

After:

In [ ]:
from address_cleaning.functions.pre_processing import process_and_deduplicate_address

deduplicated_df = process_and_deduplicate_address(df, "_c1", similarity_threshold = 95, create_flag = True, overwrite = False)
deduplicated_df.filter(deduplicated_df.words_deduplicated_flag == 1).show(truncate = False)

### Deduplicate Postcodes (REVIEW CHANGES)

Detects and removes duplicated patterns across both standard and irregular formatted postcodes

Before:

In [ ]:
from address_cleaning.functions.pre_processing import deduplicate_postcodes_udf

deduplicate_postcodes_udf = deduplicate_postcodes_udf()
deduplicated_postcodes = deduplicate_postcodes_udf(df["_c1"])["final_cleaned_address"]
deduplicated_postcodes_flag = deduplicate_postcodes_udf(df["_c1"])["changes_flag"]

deduplicate_postcodes_df = df.withColumn("cleaned_postcode", deduplicated_postcodes)
deduplicate_postcodes_df = deduplicate_postcodes_df.withColumn("postcode_deduplicated_flag", deduplicated_postcodes_flag)

deduplicate_postcodes_df.filter(deduplicate_postcodes_df.postcode_deduplicated_flag == 1).show(truncate = False)

After:

In [ ]:
from address_cleaning.functions.pre_processing import deduplicate_postcodes

deduplicated_postcodes_df = deduplicate_postcodes(df, "_c1", create_flag = True, overwrite = False)
deduplicated_postcodes_df.filter(deduplicated_postcodes_df.postcode_deduplicated_flag == 1).show(truncate = False)

### Postcode Validation and Correction

Corrects and validates UK postcodes within an address string by applying character mapping to fix common misinterpretations (e.g., 'I' to '1') and checks if the resulting postcode is valid.



In [ ]:
from address_cleaning.functions.pre_processing import map_and_check_postcode

map_and_check_postcode_df = map_and_check_postcode(df, '_c1', create_flag = True, overwrite = False)
map_and_check_postcode_df.filter(map_and_check_postcode_df.postcode_mapping_flag == 1).show(truncate = False)

### Standardise Street Types 

Standardises street type abbreviations and common misspellings within an address column, applying a set of predefined rules to replace short forms like 'ST' with 'STREET' and fix common typos.

In [ ]:
from address_cleaning.functions.pre_processing import standardise_street_types

standardised_street_types_df = standardise_street_types(df, '_c1', create_flag = True, overwrite = True)
standardised_street_types_df.filter(standardised_street_types_df.street_type_standardised_flag == 1).show(truncate = False)

## Quality Flags Examples

### Validate Address Components (REVIEW)

Flags DataFrame records that contain only a town name from a predefined list and a valid UK postcode, excluding the special case 'ZZ99'. It leverages fuzzy matching to determine if the town name in each address matches any name in a predefined list above the specified degree of similarity.

Updates: 
- New function abstracted to validate components against any of the comparator lists in address_cleaning.resources.
- Adaptable to various levels of similarity (adjustable in parameters).
- No need for 4/5 seperate functions when can have 1 adaptable function.
- Removed postcode filtering, would make sense to have this as a seperate function.
- General bug fixes

Problems:
- Running df.filter(____).show(truncate = False) seems to be causing issues. Looks to be in the filtering stage where 'timeout' error is being thrown. Dropping the .filter() seems to work fine.

In [ ]:
from address_cleaning.functions.quality_flags import validate_address_components
from address_cleaning.resources import town_list

validate_components_df = validate_address_components(df, '_c1', flag_suffix = 'town_flag', comparator_list=town_list, similarity_threshold=90)
validate_components_df = validate_components_df.show(truncate = False)

In [ ]:
from address_cleaning.resources import county_list
validate_components_df = validate_address_components(df, '_c1', flag_suffix = 'county_flag', comparator_list=county_list, similarity_threshold=90)
validate_components_df = validate_components_df.show(truncate = False)

In [ ]:
from address_cleaning.resources import disallowed_country_list

validate_components_df = validate_address_components(df, '_c1', flag_suffix = 'disallowed_country_flag', comparator_list=disallowed_country_list, similarity_threshold=90)
validate_components_df = validate_components_df.show(truncate = False)